<a href="https://colab.research.google.com/github/apar-tech/rag-chatbot/blob/main/phase2/query.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
try:
    !pip install -q \
    chromadb \
    google-genai \
    groq

    print("Libraries installed successfully!")

except Exception as e:
    print("Installation Error:")
    print(e)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not current

In [ ]:
import os

print("chunks.pkl size:", os.path.getsize("chunks.pkl"))
print("chunk_embeddings.pkl size:", os.path.getsize("chunk_embeddings.pkl"))

chunks.pkl size: 198693
chunk_embeddings.pkl size: 3070468


In [ ]:
try:
    import pickle
    import chromadb

    # Load saved data
    with open("chunks.pkl", "rb") as f:
        chunks = pickle.load(f)

    with open("chunk_embeddings.pkl", "rb") as f:
        chunk_embeddings = pickle.load(f)

    print("Files loaded successfully!")

    # Create ChromaDB client
    chroma_client = chromadb.PersistentClient(
        path="./digital_chromadb"
    )

    # Create collection
    collection = chroma_client.get_or_create_collection(
        name="digital_docs"
    )

    # Add data to collection
    collection.add(
        ids=[f"chunk_{i}" for i in range(len(chunks))],
        documents=chunks,
        embeddings=chunk_embeddings
    )

    print("Collection created successfully!")
    print("Records:", collection.count())

except Exception as e:
    print("Error:")
    print(e)

Files loaded successfully!
Collection created successfully!
Records: 111


In [ ]:
try:

    from google import genai
    from google.colab import userdata

    client = genai.Client(
        api_key=userdata.get("geminikey")
    )

    print("Gemini initialized successfully!")

except Exception as e:

    print("Gemini Error:")
    print(e)

Gemini initialized successfully!


In [ ]:
try:

    query = "What is Artificial intelligence?"

    response = client.models.embed_content(
        model="models/gemini-embedding-001",
        contents=query
    )

    query_embedding = response.embeddings[0].values

    print("Query embedding generated!")

except Exception as e:

    print("Embedding Error:")
    print(e)

Query embedding generated!


In [ ]:
try:

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=5
    )

    print("Retrieved top 5 chunks!")

    retrieved_chunks = results["documents"][0]

    for i, chunk in enumerate(retrieved_chunks):

        print("=" * 60)
        print(f"Chunk {i + 1}")
        print(chunk[:500])
        print()

except Exception as e:
  print("Error:",e)

Retrieved top 5 chunks!
Chunk 1
Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.
High-pr

Chunk 2
Russell and Norvig agree with Turing that intelligence must be defined in terms of external behavior, not internal structure. However, they are critical that the test requires the machine to imitate humans. "Aeronautical engineering texts", they wrote, "do not define the goal of their field as making 'machines that fly so exactly like pigeons that they can fool other pigeons.'" AI founder John McCarthy agreed, writing that "Artificial intelligence is not

In [ ]:
try:

    from groq import Groq
    from google.colab import userdata

    groq_client = Groq(
        api_key=userdata.get("groqkey")
    )

    print("Groq initialized successfully!")
    context = "\n\n".join(retrieved_chunks)

    print("Context created successfully!")

except Exception as e:

    print("Groq Error:")
    print(e)



Groq initialized successfully!
Context created successfully!


In [ ]:

try:

    prompt = f"""
You are a networking assistant.

Answer only using the provided context.

If the answer is not found, reply:

I could not find that information in the knowledge base.

Context:
{context}

Question:
{query}
"""

    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "user",
                "content": prompt
            }       ]
    )

    answer = response.choices[0].message.content

    print(answer)

except Exception as e:

    print("Generation Error:")
    print(e)

Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.


In [ ]:
def ask_network_question(question):

    try:

        embedding_response = client.models.embed_content(
            model="models/gemini-embedding-001",
            contents=question
        )

        query_embedding = embedding_response.embeddings[0].values

        search_results = collection.query(
            query_embeddings=[query_embedding],
            n_results=5
        )

        context = "\n\n".join(
            search_results["documents"][0]
        )

        prompt = f"""
You are a networking assistant.

Answer only using the provided context.

Context:
{context}
Question:
{question}
"""

        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        )

        return response.choices[0].message.content

    except Exception as e:

        return f"Error: {e}"

In [ ]:

print(
    ask_network_question(
        "What is the purpose of Machine learning?"
    )
)

The purpose of Machine Learning (ML) is to develop and study statistical algorithms that can learn from data and generalize to unseen data, allowing them to perform tasks without being explicitly programmed.


In [ ]:

print(
    ask_network_question(
        "Explain machine learning,deep learning and Artificial intelligence?"
    )
)

Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, thus performing tasks without being explicitly programmed. 

Deep learning (DL) is a type of machine learning that focuses on utilizing multilayered neural networks to perform tasks such as classification, regression, and representation learning. It takes inspiration from biological neuroscience and revolves around stacking artificial neurons into layers and "training" them to process data.

Artificial intelligence (AI) is the broader field that encompasses machine learning and deep learning. Machine learning is the study of programs that can improve their performance on a given task automatically, and it has been a part of AI from the beginning. AI involves the development of algorithms and statistical models that enable machines to perform tasks that typically require human intelligence, such as